# Guider centroid check — red dot vs independent centroid, per stamp

**Purpose:** verify the summit_utils guider red-dot (`xroi`,`yroi`) sits on the star,
by overlaying it against an independent flux center-of-mass on each stamp of one
exposure/detector, in the same zoomed DVCS frame the RubinTV movie uses.

**Author:** A. Roodman  ·  **Status:** diagnostic

For (`day_obs`,`seq_num`,`detector`): recompute the tracker inline, then show the
coadd and each stamp zoomed on the fixed reference center, with the tracker centroid
(red ●) and an independent COM (green +). A summary plots (tracker − COM) vs stamp.

## Contents
1. [Parameters](#params)
2. [Setup & tracking](#setup)
3. [Coadd](#coadd)
4. [Per-stamp panels](#stamps)
5. [Tracker vs COM over stamps](#summary)

<a id='params'></a>
## 1. Parameters

In [ ]:
day_obs = 20260706
seq_num = 44
detector = 'R00_SG1'

cutout_size = 20          # zoom box (pixels); matches the RubinTV star_movie
com_half = 6              # half-window (pixels) for the independent COM
n_panels = 20             # number of stamps to show (evenly spaced across the exposure)
plo, phi = 50, 98         # display stretch percentiles (RubinTV star_movie)

repo = 'main'
collections = ['LSSTCam/raw/guider', 'LSSTCam/raw/all']

<a id='setup'></a>
## 2. Setup & tracking

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, 'code')
from guiderEdgeRecovery import makeTrackerConfig
from lsst.daf.butler import Butler
from lsst.summit.utils.guiders.reading import GuiderReader
from lsst.summit.utils.guiders.tracking import GuiderStarTracker

reader = GuiderReader(Butler(repo, collections=collections), view='dvcs')
gd = reader.get(dayObs=day_obs, seqNum=seq_num, doSubtractMedian=True)
cfg = makeTrackerConfig(minFiniteFraction=0.5, minSnr=10.0, maxEllipticity=0.7, edgeMargin=3)
stars = GuiderStarTracker(gd, cfg).trackGuiderStars(refCatalog=None)
sdf = stars[stars['detector'] == detector].sort_values('stamp').reset_index(drop=True)
assert not sdf.empty, f'no tracked stars for {detector}'
refX = float(sdf['xroi_ref'].median()); refY = float(sdf['yroi_ref'].median())
print(f'{detector}: {len(sdf)} stamps tracked | refCenter=({refX:.2f},{refY:.2f})')

### Helpers

In [ ]:
def com_window(img, cx, cy, half):
    """Background-subtracted flux center-of-mass in a +/-half window (0-based)."""
    ny, nx = img.shape
    x0, x1 = max(0, int(round(cx)) - half), min(nx, int(round(cx)) + half + 1)
    y0, y1 = max(0, int(round(cy)) - half), min(ny, int(round(cy)) + half + 1)
    sub = img[y0:y1, x0:x1].astype(float) - np.nanmedian(img)
    sub[~np.isfinite(sub)] = 0.0; sub[sub < 0] = 0.0
    ys, xs = np.mgrid[y0:y1, x0:x1]
    tot = sub.sum()
    if tot <= 0:
        return np.nan, np.nan
    return float((xs * sub).sum() / tot), float((ys * sub).sum() / tot)

def show_zoom(ax, img, cx, cy, dotxy, comxy, size, title=None):
    """Zoomed panel centered on (cx,cy) with tracker dot + COM marker."""
    h, w = img.shape; half = size // 2
    x0, x1 = max(0, int(cx) - half), min(w, int(cx) + half)
    y0, y1 = max(0, int(cy) - half), min(h, int(cy) + half)
    reg = img[y0:y1, x0:x1]
    vmin, vmax = np.nanpercentile(reg, [plo, phi])
    ax.imshow(reg, origin='lower', cmap='Greys', vmin=vmin, vmax=vmax,
              extent=(x0, x1, y0, y1), interpolation='nearest')
    if np.isfinite(dotxy[0]):
        ax.plot(*dotxy, 'o', color='firebrick', ms=7, label='tracker xroi')
    if np.isfinite(comxy[0]):
        ax.plot(*comxy, '+', color='limegreen', ms=11, mew=2.2, label='flux COM')
    ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy - half, cy + half)
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=8)

<a id='coadd'></a>
## 3. Coadd

Red ● = tracker `refCenter` (median `xroi`); green + = COM of the coadd.

In [ ]:
coadd = gd.getStampArrayCoadd(detector)
cmx, cmy = com_window(coadd, refX, refY, com_half)
fig, ax = plt.subplots(figsize=(6, 6))
show_zoom(ax, coadd, refX, refY, (refX, refY), (cmx, cmy), cutout_size,
          title=f'{detector} coadd  {day_obs}/{seq_num}')
ax.legend(loc='upper right', fontsize=8)
print(f'coadd: tracker refCenter=({refX:.2f},{refY:.2f})  COM=({cmx:.2f},{cmy:.2f})  '
      f'diff=({refX-cmx:+.2f},{refY-cmy:+.2f}) px')

<a id='stamps'></a>
## 4. Per-stamp panels

Each panel is one stamp, zoomed on the fixed reference center (as the movie).
Red ● = that stamp's tracker (`xroi`,`yroi`); green + = independent COM.

In [ ]:
idx = np.linspace(0, len(sdf) - 1, min(n_panels, len(sdf))).round().astype(int)
idx = sorted(set(idx))
ncol = 5; nrow = int(np.ceil(len(idx) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax in axes[len(idx):]:
    ax.set_visible(False)
for ax, i in zip(axes, idx):
    row = sdf.iloc[i]
    st = int(row['stamp'])
    xr, yr = float(row['xroi']), float(row['yroi'])
    img = gd[detector, st]
    cmx, cmy = com_window(img, xr, yr, com_half)
    show_zoom(ax, img, refX, refY, (xr, yr), (cmx, cmy), cutout_size,
              title=f'stamp {st}  d=({xr-cmx:+.1f},{yr-cmy:+.1f})')
axes[0].legend(loc='upper right', fontsize=7)
fig.suptitle(f'{detector}  {day_obs}/{seq_num}  — tracker ● vs COM +  (title d = tracker-COM px)')
fig.tight_layout()

<a id='summary'></a>
## 5. Tracker vs COM over all stamps

In [ ]:
dx, dy, sn = [], [], []
for _, row in sdf.iterrows():
    img = gd[detector, int(row['stamp'])]
    cmx, cmy = com_window(img, float(row['xroi']), float(row['yroi']), com_half)
    dx.append(float(row['xroi']) - cmx); dy.append(float(row['yroi']) - cmy)
    sn.append(int(row['stamp']))
dx, dy = np.array(dx), np.array(dy)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(sn, dx, '.', label='x'); ax[0].plot(sn, dy, '.', label='y')
ax[0].axhline(0, color='k', lw=0.7); ax[0].legend()
ax[0].set(xlabel='stamp', ylabel='tracker - COM [px]', title='centroid offset vs stamp')
ax[1].scatter(dx, dy, s=10, alpha=0.5); ax[1].axhline(0, color='k', lw=0.7); ax[1].axvline(0, color='k', lw=0.7)
ax[1].set_aspect('equal'); ax[1].set(xlabel='dx [px]', ylabel='dy [px]', title='offset scatter')
fig.tight_layout()
print(f'median tracker-COM: dx={np.nanmedian(dx):+.3f}  dy={np.nanmedian(dy):+.3f} px  '
      f'(rms {np.nanstd(dx):.3f}, {np.nanstd(dy):.3f})')